In [0]:
%python
dbutils.widgets.text("catalog","default")
catalog=dbutils.widgets.get("catalog")

In [0]:
create table if not exists IDENTIFIER(:catalog || '.gold.dim_date') (
  date_key       INT,
  calendar_date  DATE,
  day_name       STRING,
  iso_week       INT,
  is_weekend     BOOLEAN
)
using delta;

merge into IDENTIFIER(:catalog || '.gold.dim_date') t
using (
  select
    int(date_format(d, 'yyyyMMdd')) as date_key,
    d                               as calendar_date,
    date_format(d, 'EEEE')          as day_name,
    int(weekofyear(d))              as iso_week,
    dayofweek(d) in (1, 7)          as is_weekend
  from (select explode(sequence(date '2026-03-01', date '2026-03-31', interval 1 day)) as d)
) s
on t.date_key = s.date_key
when matched and (
  t.calendar_date is distinct from s.calendar_date
  or t.day_name is distinct from s.day_name
  or t.iso_week is distinct from s.iso_week
  or t.is_weekend is distinct from s.is_weekend
) then update set *
when not matched then insert *

In [0]:
create table if not exists IDENTIFIER(:catalog || '.gold.dim_store') (
  store_id      STRING,
  store_name    STRING,
  city          STRING,
  state         STRING,
  region        STRING,
  opening_date  DATE
)
using delta;

merge into IDENTIFIER(:catalog || '.gold.dim_store') t
using (
  select store_id, store_name, city, state, region, opening_date
  from IDENTIFIER(:catalog || '.silver.stores')
) s
on t.store_id = s.store_id
when matched and (
  t.store_name is distinct from s.store_name
  or t.city is distinct from s.city
  or t.state is distinct from s.state
  or t.region is distinct from s.region
  or t.opening_date is distinct from s.opening_date
) then update set *
when not matched then insert *

In [0]:
create table if not exists IDENTIFIER(:catalog || '.gold.dim_product') (
  product_id     STRING,
  name           STRING,
  brand          STRING,
  category_l1    STRING,
  category_l2    STRING,
  pack_size      INT,
  pack_unit      STRING,
  mrp            DECIMAL(12,2),
  selling_price  DECIMAL(12,2),
  is_active      BOOLEAN
)
using delta;

merge into IDENTIFIER(:catalog || '.gold.dim_product') t
using (
  select product_id, name, brand, category_l1, category_l2, pack_size, pack_unit, mrp, selling_price, is_active
  from IDENTIFIER(:catalog || '.silver.products')
) s
on t.product_id = s.product_id
when matched and (
  t.name is distinct from s.name
  or t.brand is distinct from s.brand
  or t.category_l1 is distinct from s.category_l1
  or t.category_l2 is distinct from s.category_l2
  or t.pack_size is distinct from s.pack_size
  or t.pack_unit is distinct from s.pack_unit
  or t.mrp is distinct from s.mrp
  or t.selling_price is distinct from s.selling_price
  or t.is_active is distinct from s.is_active
) then update set *
when not matched then insert *

In [0]:
create table if not exists IDENTIFIER(:catalog || '.gold.dim_customer') (
  customer_sk    BIGINT UNIQUE NOT NULL,
  customer_id    STRING,
  full_name      STRING,
  email          STRING,
  phone          STRING,
  city           STRING,
  tier           STRING,
  signup_date    DATE,
  effective_from DATE,
  effective_to   DATE,
  is_current     BOOLEAN
)
using delta;

merge into IDENTIFIER(:catalog || '.gold.dim_customer') t
using (
  select
    xxhash64(customer_id, effective_from) as customer_sk,
    customer_id, full_name, email, phone, city, tier, signup_date, effective_from, effective_to, is_current
  from IDENTIFIER(:catalog || '.silver.customers')
) s
on t.customer_sk = s.customer_sk
when matched and (
  t.customer_id is distinct from s.customer_id
  or t.full_name is distinct from s.full_name
  or t.email is distinct from s.email
  or t.phone is distinct from s.phone
  or t.city is distinct from s.city
  or t.tier is distinct from s.tier
  or t.signup_date is distinct from s.signup_date
  or t.effective_from is distinct from s.effective_from
  or t.effective_to is distinct from s.effective_to
  or t.is_current is distinct from s.is_current
) then update set *
when not matched then insert *

In [0]:
create table if not exists IDENTIFIER(:catalog || '.gold.fact_orders') (
  order_id         STRING,
  order_date_key   INT,
  customer_sk      BIGINT,
  customer_id      STRING,
  store_id         STRING,
  order_status     STRING,
  payment_method   STRING,
  channel          STRING,
  line_count       INT,
  units            INT,
  gross_amount     DECIMAL(12,2),
  discount_amount  DECIMAL(12,2),
  net_item_amount  DECIMAL(12,2),
  delivery_fee     DECIMAL(10,2),
  order_total      DECIMAL(12,2),
  promised_mins    INT,
  actual_mins      INT,
  is_on_time       BOOLEAN
)
using delta;

merge into IDENTIFIER(:catalog || '.gold.fact_orders') t
using (
  with order_agg as (
    select
      order_id,
      count(*)             as line_count,
      sum(quantity)        as units,
      sum(gross_amount)    as gross_amount,
      sum(discount_amount) as discount_amount,
      sum(net_amount)      as net_item_amount
    from IDENTIFIER(:catalog || '.silver.order_items')
    group by order_id
  )
  select
    o.order_id,
    int(date_format(o.order_date, 'yyyyMMdd')) as order_date_key,
    dc.customer_sk,
    o.customer_id,
    o.store_id,
    o.order_status,
    o.payment_method,
    cast(null as string)         as channel,
    coalesce(agg.line_count, 0)  as line_count,
    coalesce(agg.units, 0)       as units,
    cast(agg.gross_amount as decimal(12,2))     as gross_amount,
    cast(agg.discount_amount as decimal(12,2))  as discount_amount,
    cast(agg.net_item_amount as decimal(12,2))  as net_item_amount,
    cast(o.delivery_fee as decimal(10,2))        as delivery_fee,
    cast(agg.net_item_amount + o.delivery_fee as decimal(12,2)) as order_total,
    o.promised_mins,
    o.actual_mins,
    case
      when o.order_status = 'DELIVERED' then o.actual_mins <= o.promised_mins
      else null
    end as is_on_time
  from IDENTIFIER(:catalog || '.silver.orders') o
  left join order_agg agg on o.order_id = agg.order_id
  left join IDENTIFIER(:catalog || '.gold.dim_customer') dc
    on o.customer_id = dc.customer_id
    and dc.effective_from <= o.order_date
    and (dc.effective_to is null or o.order_date < dc.effective_to)
) s
on t.order_id = s.order_id
when matched and (
  t.order_date_key is distinct from s.order_date_key
  or t.customer_sk is distinct from s.customer_sk
  or t.customer_id is distinct from s.customer_id
  or t.store_id is distinct from s.store_id
  or t.order_status is distinct from s.order_status
  or t.payment_method is distinct from s.payment_method
  or t.channel is distinct from s.channel
  or t.line_count is distinct from s.line_count
  or t.units is distinct from s.units
  or t.gross_amount is distinct from s.gross_amount
  or t.discount_amount is distinct from s.discount_amount
  or t.net_item_amount is distinct from s.net_item_amount
  or t.delivery_fee is distinct from s.delivery_fee
  or t.order_total is distinct from s.order_total
  or t.promised_mins is distinct from s.promised_mins
  or t.actual_mins is distinct from s.actual_mins
  or t.is_on_time is distinct from s.is_on_time
) then update set *
when not matched then insert *

In [0]:
create table if not exists IDENTIFIER(:catalog || '.gold.fact_order_line') (
  order_id         STRING,
  line_number      INT,
  order_date_key   INT,
  customer_sk      BIGINT,
  store_id         STRING,
  product_id       STRING,
  order_status     STRING,
  quantity         INT,
  unit_price       DECIMAL(12,2),
  gross_amount     DECIMAL(12,2),
  discount_amount  DECIMAL(12,2),
  net_amount       DECIMAL(12,2)
)
using delta;

merge into IDENTIFIER(:catalog || '.gold.fact_order_line') t
using (
  select
    oi.order_id,
    oi.line_number,
    int(date_format(o.order_date, 'yyyyMMdd')) as order_date_key,
    dc.customer_sk,
    o.store_id,
    oi.product_id,
    o.order_status,
    oi.quantity,
    cast(oi.unit_price as decimal(12,2))      as unit_price,
    cast(oi.gross_amount as decimal(12,2))    as gross_amount,
    cast(oi.discount_amount as decimal(12,2))  as discount_amount,
    cast(oi.net_amount as decimal(12,2))      as net_amount
  from IDENTIFIER(:catalog || '.silver.order_items') oi
  left join IDENTIFIER(:catalog || '.silver.orders') o on oi.order_id = o.order_id
  left join IDENTIFIER(:catalog || '.gold.dim_customer') dc
    on o.customer_id = dc.customer_id
    and dc.effective_from <= o.order_date
    and (dc.effective_to is null or o.order_date < dc.effective_to)
) s
on t.order_id = s.order_id and t.line_number = s.line_number
when matched and (
  t.order_date_key is distinct from s.order_date_key
  or t.customer_sk is distinct from s.customer_sk
  or t.store_id is distinct from s.store_id
  or t.product_id is distinct from s.product_id
  or t.order_status is distinct from s.order_status
  or t.quantity is distinct from s.quantity
  or t.unit_price is distinct from s.unit_price
  or t.gross_amount is distinct from s.gross_amount
  or t.discount_amount is distinct from s.discount_amount
  or t.net_amount is distinct from s.net_amount
) then update set *
when not matched then insert *

In [0]:
create table if not exists IDENTIFIER(:catalog || '.gold.agg_daily_store_sales') (
  order_date           DATE,
  store_id             STRING,
  city                 STRING,
  delivered_orders     INT,
  cancelled_orders     INT,
  units_sold           INT,
  gross_amount         DECIMAL(12,2),
  discount_amount      DECIMAL(12,2),
  net_item_revenue     DECIMAL(12,2),
  delivery_fee_revenue DECIMAL(12,2),
  on_time_orders       INT,
  on_time_rate_pct    DECIMAL(5,2)
)
using delta;

merge into IDENTIFIER(:catalog || '.gold.agg_daily_store_sales') t
using (
  with date_range as (
    select
      min(dd.calendar_date) as min_date,
      max(dd.calendar_date) as max_date
    from IDENTIFIER(:catalog || '.gold.fact_orders') fo
    inner join IDENTIFIER(:catalog || '.gold.dim_date') dd on fo.order_date_key = dd.date_key
  ),
  all_dates as (
    select explode(sequence(
      (select min_date from date_range),
      (select max_date from date_range),
      interval 1 day
    )) as order_date
  ),
  store_sales as (
    select
      dd.calendar_date as order_date,
      fo.store_id,
      cast(sum(case when fo.order_status = 'DELIVERED' then 1 else 0 end) as int) as delivered_orders,
      cast(sum(case when fo.order_status = 'CANCELLED' then 1 else 0 end) as int) as cancelled_orders,
      cast(sum(case when fo.order_status = 'DELIVERED' then fo.units else 0 end) as int) as units_sold,
      cast(sum(case when fo.order_status = 'DELIVERED' then fo.gross_amount else 0 end) as decimal(12,2)) as gross_amount,
      cast(sum(case when fo.order_status = 'DELIVERED' then fo.discount_amount else 0 end) as decimal(12,2)) as discount_amount,
      cast(sum(case when fo.order_status = 'DELIVERED' then fo.net_item_amount else 0 end) as decimal(12,2)) as net_item_revenue,
      cast(sum(case when fo.order_status = 'DELIVERED' then fo.delivery_fee else 0 end) as decimal(12,2)) as delivery_fee_revenue,
      cast(sum(case when fo.order_status = 'DELIVERED' and fo.is_on_time then 1 else 0 end) as int) as on_time_orders
    from IDENTIFIER(:catalog || '.gold.fact_orders') fo
    inner join IDENTIFIER(:catalog || '.gold.dim_date') dd on fo.order_date_key = dd.date_key
    group by dd.calendar_date, fo.store_id
  )
  select
    d.order_date,
    ds.store_id,
    ds.city,
    coalesce(ss.delivered_orders, 0)      as delivered_orders,
    coalesce(ss.cancelled_orders, 0)        as cancelled_orders,
    coalesce(ss.units_sold, 0)             as units_sold,
    coalesce(ss.gross_amount, 0)            as gross_amount,
    coalesce(ss.discount_amount, 0)        as discount_amount,
    coalesce(ss.net_item_revenue, 0)       as net_item_revenue,
    coalesce(ss.delivery_fee_revenue, 0)   as delivery_fee_revenue,
    coalesce(ss.on_time_orders, 0)          as on_time_orders,
    case
      when coalesce(ss.delivered_orders, 0) = 0 then null
      else cast(round(
        ss.on_time_orders * 100.0 / ss.delivered_orders,
        2
      ) as decimal(5,2))
    end as on_time_rate_pct
  from all_dates d
  cross join IDENTIFIER(:catalog || '.gold.dim_store') ds
  left join store_sales ss on d.order_date = ss.order_date and ds.store_id = ss.store_id
) s
on t.order_date = s.order_date and t.store_id = s.store_id
when matched and (
  t.city is distinct from s.city
  or t.delivered_orders is distinct from s.delivered_orders
  or t.cancelled_orders is distinct from s.cancelled_orders
  or t.units_sold is distinct from s.units_sold
  or t.gross_amount is distinct from s.gross_amount
  or t.discount_amount is distinct from s.discount_amount
  or t.net_item_revenue is distinct from s.net_item_revenue
  or t.delivery_fee_revenue is distinct from s.delivery_fee_revenue
  or t.on_time_orders is distinct from s.on_time_orders
  or t.on_time_rate_pct is distinct from s.on_time_rate_pct
) then update set *
when not matched then insert *

In [0]:
create or replace view IDENTIFIER(:catalog || '.gold.v_kpi_summary') as
with order_kpis as (
  select
    cast(sum(case when order_status = 'DELIVERED' then 1 else 0 end) as int) as delivered_orders,
    cast(sum(case when order_status = 'CANCELLED' then 1 else 0 end) as int) as cancelled_orders,
    cast(sum(case when order_status = 'DELIVERED' then net_item_amount else 0 end) as decimal(12,2)) as net_item_revenue,
    cast(sum(case when order_status = 'DELIVERED' and is_on_time then 1 else 0 end) as int) as on_time_orders
  from IDENTIFIER(:catalog || '.gold.fact_orders')
),
customer_kpis as (
  select
    count(distinct customer_id) as active_customers,
    count(distinct case when delivered_cnt >= 2 then customer_id end) as repeat_customers
  from (
    select customer_id, sum(case when order_status = 'DELIVERED' then 1 else 0 end) as delivered_cnt
    from IDENTIFIER(:catalog || '.gold.fact_orders')
    group by customer_id
  )
  where delivered_cnt >= 1
)
select
  cast(ok.delivered_orders as int) as delivered_orders,
  cast(ok.cancelled_orders as int) as cancelled_orders,
  cast(ok.net_item_revenue as decimal(12,2)) as net_item_revenue,
  cast(round(ok.net_item_revenue / nullif(ok.delivered_orders, 0), 2) as decimal(12,2)) as avg_order_value,
  cast(round(ok.on_time_orders * 100.0 / nullif(ok.delivered_orders, 0), 2) as decimal(5,2)) as on_time_rate_pct,
  cast(round(ok.cancelled_orders * 100.0 / nullif(ok.cancelled_orders + ok.delivered_orders, 0), 2) as decimal(5,2)) as cancellation_rate_pct,
  cast(ck.active_customers as int) as active_customers,
  cast(round(ck.repeat_customers * 100.0 / nullif(ck.active_customers, 0), 2) as decimal(5,2)) as repeat_customer_rate_pct
from order_kpis ok
cross join customer_kpis ck